In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
import os
import pandas as pd
import sys
sys.path.append('/content/drive/...../modules')
import api_toast as toast
import json_processing_toast_catalog_extraction as catalogs



## Enter API related variables for calls

Set these before running the cell below. They come from your own Toast
account and Colab setup, not from this repo:

| Variable | Where to find it |
|---|---|
| `hostname` | Toast API subdomain for your integration, e.g. `ws-api.toasttab.com` (Toast dev portal -> API access) |
| `Client_ID` | Toast API Client ID (Toast dev portal -> your API access group -> Standard API credentials) |
| `Client_Secret` | Toast API Client Secret (same page as Client ID -- only shown once at creation, store it somewhere safe) |
| `Restaurant_GUID` | GUID of the specific restaurant location you're extracting data for (Toast dev portal -> Restaurants) |

In Google Colab: **Runtime -> Secrets** (key icon, left sidebar) -> add each
value under the exact name shown above -> grant this notebook access when
prompted. `userdata.get(...)` reads them from there. Never hardcode real
credentials directly into a cell.

In [ ]:
print("Getting Toast API hostname (e.g. ws-api.toasttab.com):")
toast_hostname = userdata.get('hostname').strip()
toast_hostname = toast_hostname.replace("https://", "").replace("http://", "").rstrip("/")

print("Getting Toast API Client ID:")
client_id = userdata.get('Client_ID').strip()

print("Getting Toast API Client Secret:")
client_secret = userdata.get('Client_Secret').strip()

print("Getting Toast API restaurant GUID:")
restaurant_guid = userdata.get('Restaurant_GUID').strip()

In [ ]:
json_output_dir = "/content/drive/...../raw_catalogs"
os.makedirs(os.path.dirname(json_output_dir), exist_ok=True)

print(f"Running Toast catalog extraction.")
print(f"Output dir: {json_output_dir}")

toast.run_catalog_extraction(toast_hostname=toast_hostname,
                             client_id=client_id,
                             client_secret=client_secret,
                             output_dir=json_output_dir,
                             restaurant_guid=restaurant_guid,)


In [ ]:
csv_output_dir = "/content/drive/...../raw_catalogs"
os.makedirs(os.path.dirname(json_output_dir), exist_ok=True)

CATALOG_BUILDERS = {
    "sales_categories": catalogs.build_dataframe_from_json_sales_category,
    "revenue_centers": catalogs.build_dataframe_from_json_revenue_centers,
    "dining_options": catalogs.build_dataframe_from_json_dining_option,
    "tables": catalogs.build_dataframe_tables,
    "menu_items": catalogs.build_dataframe_from_json_menu_items,
    "employees": catalogs.build_dataframe_from_json_employees,
    "jobs": catalogs.build_dataframe_from_json_jobs,
}


for catalog_type, builder in CATALOG_BUILDERS.items():
        json_path = os.path.join(json_output_dir, f"catalog_{catalog_type}.json")
        csv_path = os.path.join(csv_output_dir, f"catalog_{catalog_type}.csv")

        if not os.path.exists(json_path):
            print(f"[{catalog_type}] SKIPPED: {json_path} not found.")
            continue

        # Builder writes the CSV itself and returns None -- read row count
        # back from the CSV it just wrote, purely for the checkpoint print.
        builder(json_path, csv_path)
        row_count = len(pd.read_csv(csv_path))
        print(f"[{catalog_type}] {row_count} rows -> {csv_path}")
